## Exploratory Data Analysis

In [1]:
import pandas as pd

/tmp/ipykernel_3776159/4080736814.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [4]:
#payload composition & class balance for the LNP dataset


PAYLOAD_MAP = {0: "mRNA", 1: "siRNA", 2: "DNA barcode"}
LABEL_MAP = {0: "Unstable", 1: "Stable"}

df = (
    pd.read_csv("./datasets/processed_data.csv", usecols=["label", "Payload"])
    .assign(Payload=lambda d: d["Payload"].map(PAYLOAD_MAP),
            label=lambda d: d["label"].map(LABEL_MAP))
)

# Supplementary table: counts + row-normalized % per payload
counts = pd.crosstab(df["Payload"], df["label"])
pct = pd.crosstab(df["Payload"], df["label"], normalize="index").mul(100).round(1)
table = counts.astype(str) + " (" + pct.astype(str) + "%)"
table["Total"] = counts.sum(axis=1)
table = table.reindex(["mRNA", "siRNA", "DNA barcode"])
table.loc["Overall"] = (
    (counts.sum() .astype(str) + " (" + pct.mean().round(1).astype(str) + "%)").tolist()
    + [len(df)]
)

table.to_csv("./analysis/Payload_class_distribution.csv")
print(table)

# Main-text sentence
n = len(df)
print(f"\nDataset: {n} LNP formulations — "
      f"{counts.loc['mRNA'].sum()} mRNA ({counts.loc['mRNA'].sum()/n:.1%}), "
      f"{counts.loc['siRNA'].sum()} siRNA ({counts.loc['siRNA'].sum()/n:.1%}), "
      f"{counts.loc['DNA barcode'].sum()} DNA-barcoded ({counts.loc['DNA barcode'].sum()/n:.1%}). "
      f"Overall class balance: {df['label'].eq('Stable').sum()} stable : "
      f"{df['label'].eq('Unstable').sum()} unstable "
      f"({df['label'].eq('Stable').mean():.1%} stable).")

label              Stable      Unstable  Total
Payload                                       
mRNA         2014 (72.3%)   772 (27.7%)   2786
siRNA         947 (77.9%)   269 (22.1%)   1216
DNA barcode  1027 (73.8%)   365 (26.2%)   1392
Overall      3988 (74.7%)  1406 (25.3%)   5394

Dataset: 5394 LNP formulations — 2786 mRNA (51.6%), 1216 siRNA (22.5%), 1392 DNA-barcoded (25.8%). Overall class balance: 3988 stable : 1406 unstable (73.9% stable).
